# 토큰 트리밍 & 요약 메모리 — Modern 패턴

## 이번 노트북 학습 목표

- 글자 수(문자 수)를 기준으로 히스토리를 잘라내는 **토큰 트리밍 패턴**을 구현해요.
- LLM으로 오래된 대화를 압축하는 **요약 기반 메모리 패턴**을 구현해요.
- 두 패턴의 차이와 적합한 사용 시나리오를 이해해요.

## 사전 지식

- `ChatMessageHistory` 기본 사용법 (10번 노트북)
- Modern 윈도우 메모리 패턴 (12번 노트북)

## 구식 vs Modern 메모리 패턴 비교

| 항목 | 구식 (03번, `ConversationTokenBufferMemory`) | Modern 토큰 트리밍 |
|------|----------------------------------------------|-------------------|
| 메모리 클래스 | `ConversationTokenBufferMemory(max_token_limit=500)` | 없음 (함수로 대체) |
| 구현 방식 | 클래스 내부에서 자동 트리밍 | `trim_messages_by_chars()` 함수로 명시적 처리 |
| LangChain 권장 | 0.2.x 이후 deprecated | 1.0.x 권장 방식 |

| 항목 | 구식 (06번, `ConversationSummaryMemory`) | Modern 요약 메모리 |
|------|------------------------------------------|-------------------|
| 메모리 클래스 | `ConversationSummaryMemory(llm=...)` | 없음 (요약 체인으로 대체) |
| 구현 방식 | 클래스가 자동으로 요약 | `summarize_session()` 함수로 명시적 처리 |
| LangChain 권장 | 0.2.x 이후 deprecated | 1.0.x 권장 방식 |

> 이 노트북에서는 이해를 돕기 위해 **문자 수 기준**으로 구현해요. 실제 서비스에서는 `get_num_tokens()` 등 토큰 계산 유틸이나 LangGraph의 `SummarizationMiddleware`를 사용하는 것이 좋아요.


In [ ]:
from dotenv import load_dotenv

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

load_dotenv()

# LLM 초기화
model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 기본 프롬프트 (전체 히스토리 사용)
base_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 기술 지원 상담원입니다. 사용자의 문제를 차분히 도와주세요."),
        MessagesPlaceholder("chat_history"),
        ("human", "{question}"),
    ]
)

base_chain = base_prompt | model | StrOutputParser()

# 세션별 대화 기록 저장소
store: dict[str, ChatMessageHistory] = {}


def get_history(session_id: str) -> ChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]


## 1. 글자 수 기준 토큰 트리밍 패턴

`ConversationTokenBufferMemory(max_token_limit=...)`(구식 토큰 버퍼 메모리)와 유사하게, **히스토리의 총 길이가 한계를 넘으면 가장 오래된 메시지부터 제거**하는 패턴이에요.

여기서는 이해를 돕기 위해 실제 토큰 수 대신 **문자 수(글자 수)**를 기준으로 구현해요.

### 토큰 트리밍 흐름

```mermaid
flowchart TD
    QUESTION[사용자 질문] --> CHAIN[char_limited_chain 실행]
    CHAIN --> HIST[history.messages 전체 조회]
    HIST --> CHECK{총 문자 수<br/>≤ max_chars?}
    CHECK -- 예 --> USE[그대로 사용]
    CHECK -- 아니오 --> TRIM[가장 오래된 메시지 제거]
    TRIM --> CHECK
    USE --> PROMPT[base_prompt.invoke<br/>트리밍된 히스토리 주입]
    PROMPT --> LLM[ChatOpenAI]
    LLM --> SAVE[history에 전체 저장<br/>원본 보존]
    SAVE --> ANSWER[AI 응답]

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef error fill:#f8d7da,stroke:#dc3545,color:#721c24
    classDef external fill:#e2e3e5,stroke:#6c757d,color:#383d41

    class QUESTION input
    class CHAIN,CHECK,TRIM,USE,PROMPT,LLM process
    class HIST,SAVE storage
    class ANSWER output
```

> 히스토리 원본은 모두 보관하지만, 프롬프트에 들어가는 메시지는 `max_chars` 이내로 제한해요. 오래된 정보가 완전히 사라지는 단점이 있어요. 장기 맥락이 필요하다면 아래 요약 패턴을 사용하세요.


In [ ]:
# ============================================================
# TODO: 문자 수 기반 트리밍 함수와 제한 체인을 구현하세요
# 힌트: total_chars()로 전체 길이 계산 후, max_chars를 초과하면 앞에서부터 제거하세요
# 예상 결과: 오래된 대화가 제거되어 max_chars 이내로 유지되어야 합니다
# ============================================================

# ---------------------------------------------------
# 글자 수 기반 트리밍 함수 및 체인 구현
# ---------------------------------------------------

from typing import List
from langchain_core.messages import BaseMessage


def total_chars(messages: List[BaseMessage]) -> int:
    """메시지 리스트의 총 문자 수를 계산."""
    # 각 메시지 내용의 길이를 합산
    return sum(len(m.content) for m in messages)


def trim_messages_by_chars(messages: List[BaseMessage], max_chars: int) -> List[BaseMessage]:
    """총 문자 수가 max_chars를 넘지 않도록 앞에서부터 메시지를 제거.

    - 시스템 메시지 1개 + 최근 대화 몇 개만 남기는 간단한 구현
    """
    # 이미 한도 내에 있으면 그대로 반환
    if total_chars(messages) <= max_chars:
        return messages

    trimmed = list(messages)
    while trimmed and total_chars(trimmed) > max_chars:
        # 가장 오래된 메시지 하나 제거 (앞에서부터 pop)
        trimmed.pop(0)
    return trimmed


def token_like_chain(max_chars: int = 4000):
    """문자 수 한계를 가진 토큰 버퍼 체인을 생성."""

    def _chain(inputs: dict) -> str:
        session_id: str = inputs["session_id"]
        question: str = inputs["question"]

        history = get_history(session_id)

        # 1) 현재까지의 전체 메시지
        all_messages = history.messages

        # 2) 문자 수 기준으로 트리밍 (max_chars 초과 시 오래된 것 제거)
        trimmed = trim_messages_by_chars(all_messages, max_chars=max_chars)

        # 3) 트리밍된 히스토리 + 현재 질문으로 프롬프트 구성
        prompt_msg = base_prompt.invoke({
            "chat_history": trimmed,
            "question": question,
        })

        # 4) 모델 호출
        result = (model | StrOutputParser()).invoke(prompt_msg)

        # 5) 실제 히스토리에는 전체 대화를 계속 쌓음 (원본 보존)
        history.add_user_message(question)
        history.add_ai_message(result)

        return result

    return _chain


# max_chars=600: 약 600자 초과 시 오래된 메시지부터 제거
char_limited_chain = token_like_chain(max_chars=600)

In [ ]:
# ---------------------------------------------------
# 토큰 유사 트리밍 체인 테스트: 오래된 대화가 잊혀지는지 확인
# ---------------------------------------------------

tech_session = "tech_support_token_limited"

questions = [
    "안녕하세요. 소프트웨어 설치 중에 에러 코드 1722가 발생했어요.",
    "에러가 나는 단계는 설치 마법사 마지막 단계입니다.",
    "로그를 어디서 확인해야 하는지도 알려주세요.",
    "어제 이야기했던 방화벽 설정 관련 내용도 다시 설명해 주실 수 있나요?",
    "지금까지 제가 어떤 환경이라고 설명했는지도 기억하시나요?",
]


## 2. 요약 기반 메모리 패턴

`ConversationSummaryMemory(llm=...)`(구식 요약 메모리)는 클래스가 자동으로 요약을 관리했어요. Modern 패턴에서는 **요약 체인(summary chain)**을 직접 만들고, `summary_store`에 텍스트로 저장해요.

핵심 아이디어:
- 세션별로 `summary_store[session_id]`에 **지금까지의 대화 요약 텍스트**를 저장해요.
- 새 질문이 들어올 때마다 `old_summary + recent_history → LLM → new_summary`로 갱신해요.
- 프롬프트에는 "과거 요약 + 최근 몇 턴"을 함께 넣어요.

### 요약 기반 메모리 흐름

```mermaid
flowchart TD
    QUESTION[사용자 질문] --> HADD[history.add_user_message]
    HADD --> SUM_FN[summarize_session<br/>old_summary + recent 메시지]
    SUM_FN --> SUM_LLM[요약 체인<br/>summary_chain]
    SUM_LLM --> SUM_STORE[summary_store 갱신]
    SUM_STORE --> PROMPT[프롬프트 구성<br/>과거 요약 + 최근 대화 + 질문]
    PROMPT --> LLM[ChatOpenAI]
    LLM --> HADD2[history.add_ai_message]
    HADD2 --> ANSWER[AI 응답]

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef error fill:#f8d7da,stroke:#dc3545,color:#721c24
    classDef external fill:#e2e3e5,stroke:#6c757d,color:#383d41

    class QUESTION input
    class HADD,SUM_FN,SUM_LLM,PROMPT,LLM,HADD2 process
    class SUM_STORE storage
    class ANSWER output
```

> 요약을 생성하는 LLM 호출이 **추가로 발생**해요. 매 턴마다 요약 비용이 생기지만, 덕분에 히스토리가 길어져도 프롬프트 크기는 일정하게 유지돼요.

In [ ]:
# ============================================================
# TODO: 요약 기반 메모리 체인을 구현하세요
# 힌트: 매 턴마다 summary_chain으로 과거 요약을 갱신하고, 프롬프트에 요약을 주입하세요
# 예상 결과: 마지막 질문에서 AI가 초반 대화 내용을 요약으로 기억해야 합니다
# ============================================================

# ---------------------------------------------------
# 요약 기반 메모리 구현: 요약 체인 + 스토어 + 메모리 체인
# ---------------------------------------------------

# 1단계: 요약 생성에 사용할 프롬프트
# - old_summary: 이전 요약 (없으면 "(아직 요약 없음)")
# - recent_history: 최근 대화 텍스트


# summary_chain: 요약 생성 전용 체인 (매 턴마다 호출)


# 세션별 요약 저장소 (키: session_id, 값: 요약 텍스트)
summary_store: dict[str, str] = {}


def format_messages(messages: list[BaseMessage]) -> str:
    """메시지 리스트를 사람이 읽기 좋은 문자열로 변환."""
    lines = []
    for m in messages:
        role = "사용자" if m.type in ("human", "user") else "AI"
        lines.append(f"[{role}] {m.content}")
    return "\n".join(lines)


def summarize_session(session_id: str, max_recent: int = 6) -> str:
    """세션의 최근 메시지를 기존 요약과 합쳐 새로운 요약 생성."""
    pass  # TODO: 구현하세요


def summary_memory_chain(max_recent: int = 6):
    """요약 + 최근 히스토리를 함께 사용하는 체인."""

    def _chain(inputs: dict) -> str:
        pass  # TODO: 구현하세요

    return _chain


In [ ]:
# ---------------------------------------------------
# 요약 메모리 체인 테스트: 긴 대화 후 요약 활용 확인
# ---------------------------------------------------

summary_session = "tech_support_summary"

summary_questions = [
    "안녕하세요. 회사 노트북이 부팅이 안 돼서 문의드립니다.",
    "어제부터 전원 버튼을 눌러도 로고만 나오고 멈춰요.",
    "어떤 진단 절차를 먼저 진행해야 할까요?",
    "어제 알려주신 BIOS 초기화 방법을 다시 한 번 설명해 주세요.",
    "지금까지 제 장비 환경과 시도했던 조치들을 요약해서 말씀해 주실 수 있나요?",
]


## 핵심 요약

### 이번 노트북의 구현 함수

| 함수 | 역할 |
|------|------|
| `trim_messages_by_chars(messages, max_chars)` | 총 문자 수가 한계를 초과하면 오래된 메시지를 앞에서부터 제거 |
| `token_like_chain(max_chars)` | 문자 수 제한 기반 체인 생성 함수 |
| `format_messages(messages)` | 메시지 리스트를 사람이 읽기 좋은 문자열로 변환 |
| `summarize_session(session_id, max_recent)` | 과거 요약 + 최근 메시지로 새 요약 생성 및 저장 |
| `summary_memory_chain(max_recent)` | 요약 기반 메모리 체인 생성 함수 |

### 두 패턴 비교

| 항목 | 토큰 트리밍 패턴 | 요약 기반 메모리 패턴 |
|------|-----------------|---------------------|
| 장기 맥락 유지 | 불가 (오래된 메시지 제거) | 가능 (요약으로 압축 보존) |
| 구현 복잡도 | 낮음 | 높음 |
| LLM 추가 호출 | 없음 | 매 턴마다 요약 호출 발생 |
| 비용 | 낮음 | 요약 호출 비용 추가 |
| 적합한 상황 | 단기 세션, 최신 대화만 필요할 때 | 장기 세션, 초기 맥락이 중요할 때 |

### 주의사항

- 이 노트북의 예제는 **문자 수 기준**으로 구현했어요. 실제 서비스에서는 `get_num_tokens()` 또는 `tiktoken` 라이브러리를 사용해 **실제 토큰 수**를 계산하세요.
- 요약 기반 패턴은 LLM 호출 비용이 추가로 발생해요. 요약 주기를 조절(예: N턴마다 1회)하면 비용을 줄일 수 있어요.
- 요약 과정에서 세부 정보가 일부 손실될 수 있어요. 법적·규정 맥락에서는 원본 히스토리를 별도 DB에 보관하세요.
- LangChain 1.0.x 이후에는 `LangGraph`의 `SummarizationMiddleware`가 이 패턴을 더 정교하게 지원해요.

### 하이브리드 패턴 (실전 권장)

실전에서는 두 방식을 **함께** 쓰는 것이 일반적이에요.

- 최근 N턴 원문 + 과거 요약 → 프롬프트에 함께 주입
- 나머지 생(raw) 히스토리는 별도 DB, 파일, 벡터 저장소(vector store)에 보관

### 다음 단계 예고

**14번**: `InMemoryStore`와 도구(Tool)를 사용해 특정 정보(사람 이름, 선호도 등)를 **구조화하여 추출·저장**하는 엔티티 메모리(entity memory) 패턴을 학습해요.

---

## 연습 문제

### 연습 1: 하이브리드 메모리 (요약 + 최근 대화) 구현

토큰 트리밍과 요약 메모리를 **결합**하여, "과거 대화는 요약으로 압축하고, 최근 대화는 원문으로 유지"하는 하이브리드 메모리 체인을 만들어 보세요.

**요구사항:**
1. 세션의 전체 대화 중 **최근 4개 메시지는 원문으로 유지**하고, 나머지 오래된 메시지는 **LLM으로 요약**하세요
2. 프롬프트에는 `[과거 요약]`과 `[최근 대화(원문)]`을 모두 포함하세요
3. 아래 시나리오로 테스트하세요 (총 6개 질문):
   - "안녕하세요. 저는 마케팅팀 이영희입니다."
   - "우리 팀은 다음 달에 신제품 런칭 캠페인을 준비하고 있어요."
   - "예산은 5천만원이고, 타깃은 2030 여성입니다."
   - "SNS 마케팅과 인플루언서 협업을 병행할 계획이에요."
   - "경쟁사 분석 결과도 공유드릴게요. A사는 가격, B사는 디자인이 강점이에요."
   - "지금까지 제가 공유한 캠페인 정보를 모두 요약해 주세요."
4. 마지막 질문에서 "과거 요약 + 최근 원문"을 모두 활용한 응답이 나오는지 확인하세요

**힌트:**
- 이미 노트북에 구현된 `summarize_session`과 `format_messages` 함수를 재활용하세요
- 프롬프트 구성 시 `system` 메시지에 요약 텍스트를, `MessagesPlaceholder`나 별도 변수에 최근 원문을 넣을 수 있습니다

In [ ]:
# ============================================================
# TODO: 하이브리드 메모리 체인(요약 + 최근 원문)을 구현하세요
# 힌트: 최근 recent_count개는 원문으로, 나머지는 요약으로 처리하세요
# 예상 결과: 마지막 질문에서 초반 정보(요약)와 최근 정보(원문)를 모두 활용해야 합니다
# ============================================================

# ---------------------------------------------------
# 연습 1 풀이: 하이브리드 메모리 (요약 + 최근 원문) 구현
# ---------------------------------------------------

# 마케팅 캠페인 시나리오 테스트
campaign_questions = [
    "안녕하세요. 저는 마케팅팀 이영희입니다.",
    "우리 팀은 다음 달에 신제품 런칭 캠페인을 준비하고 있어요.",
    "예산은 5천만원이고, 타깃은 2030 여성입니다.",
    "SNS 마케팅과 인플루언서 협업을 병행할 계획이에요.",
    "경쟁사 분석 결과도 공유드릴게요. A사는 가격, B사는 디자인이 강점이에요.",
    "지금까지 제가 공유한 캠페인 정보를 모두 요약해 주세요.",
]
